Funcion quantized_po2

In [18]:
import numpy as np
import tensorflow as tf
from qkeras import quantizers

# -------------------------
# CONFIG
# -------------------------
bits = 3
#use_stochastic_rounding = False POR DEFECTO

# Valores de prueba
x = np.array([12.44], dtype=np.float32)

# -------------------------
# 1) Crear cuantizador QKeras
# -------------------------
mv=None
q = quantizers.quantized_po2(bits,max_value=mv)

print("===================================")
print("QKeras quantized_po2")
print("===================================")
print(f"bits = {bits}")
# -------------------------
# 2) Reglas internas (según QKeras)
#    - QKeras separa signo => queda non_sign_bits = bits - 1
#    - si max_value=None, asume que hace falta bit de signo del exponente (need_exponent_sign_bit = 1)
#    - effect_bits = non_sign_bits - need_exponent_sign_bit
#    - k_min = -2^(effect_bits)
#    - k_max =  2^(effect_bits) - 1
# -------------------------
non_sign_bits = bits - 1
need_exponent_sign_bit = 1  # en QKeras, cuando max_value=None => True (asume exponente con signo)
effect_bits = non_sign_bits - need_exponent_sign_bit

k_min = -(2 ** effect_bits)
k_max = (2 ** effect_bits) - 1

print("Reglas internas (max_value=None):")
print(f"non_sign_bits = bits - 1 = {non_sign_bits}")
print(f"need_exponent_sign_bit = {need_exponent_sign_bit}")
print(f"effect_bits = non_sign_bits - need_exponent_sign_bit = {effect_bits}")
print(f"k_min = -2^(effect_bits) = {k_min}")
print(f"k_max =  2^(effect_bits) - 1 = {k_max}")
print("-----------------------------------")

# -------------------------
# 3) Exponentes y valores posibles (teóricos)
# -------------------------
k_list = list(range(k_min, k_max + 1))
pos_vals = [float(2.0 ** k) for k in k_list]
all_vals = sorted([-v for v in pos_vals] + pos_vals)

print(f"Exponentes posibles k = {k_list}")
print(f"Valores positivos posibles = {pos_vals}")
print(f"Valores posibles con signo = {all_vals}")
print("-----------------------------------")

# -------------------------
# 4) Cuantización "por fórmula" (equivalente a QKeras cuando no hay stochastic rounding)
#    k = round(log2(|x|))
#    k_clipped = clip(k, k_min, k_max)
#    x_q = sign(x) * 2^(k_clipped)
# -------------------------
x_abs = np.abs(x)
k_raw = np.round(np.log2(x_abs)).astype(np.int32)
k_clip = np.clip(k_raw, k_min, k_max)
xq_formula = np.sign(x) * (2.0 ** k_clip)

# Salida real QKeras
xq_qkeras = q(tf.constant(x)).numpy()

print("Entradas x:")
print(x)

print("\nPaso a paso:")
print("abs(x):")
print(x_abs)

print("\nk_raw = round(log2(|x|)):")
print(k_raw)

print(f"\nk_clipped = clip(k_raw, {k_min}, {k_max}):")
print(k_clip)

print("\nx_q (formula) = sign(x) * 2^(k_clipped):")
print(xq_formula)

print("\nx_q (QKeras):")
print(xq_qkeras)

print("\n¿Coinciden formula y QKeras?")
print(np.allclose(xq_formula, xq_qkeras))
print("===================================")

QKeras quantized_po2
bits = 3
Reglas internas (max_value=None):
non_sign_bits = bits - 1 = 2
need_exponent_sign_bit = 1
effect_bits = non_sign_bits - need_exponent_sign_bit = 1
k_min = -2^(effect_bits) = -2
k_max =  2^(effect_bits) - 1 = 1
-----------------------------------
Exponentes posibles k = [-2, -1, 0, 1]
Valores positivos posibles = [0.25, 0.5, 1.0, 2.0]
Valores posibles con signo = [-2.0, -1.0, -0.5, -0.25, 0.25, 0.5, 1.0, 2.0]
-----------------------------------
Entradas x:
[12.44]

Paso a paso:
abs(x):
[12.44]

k_raw = round(log2(|x|)):
[4]

k_clipped = clip(k_raw, -2, 1):
[1]

x_q (formula) = sign(x) * 2^(k_clipped):
[2.]

x_q (QKeras):
[2.]

¿Coinciden formula y QKeras?
True


In [19]:
from qkeras import quantizers

bits = 3

for mv in [None, 0.5, 1.0, 2.0, 8.0]:

    if mv is None:
        q = quantizers.quantized_po2(bits)
        need_exponent_sign_bit = 1
    else:
        q = quantizers.quantized_po2(bits, max_value=mv)
        need_exponent_sign_bit = 1 if mv > 1 else 0

    print(
        "max_value =", mv,
        "-> need_exponent_sign_bit =", need_exponent_sign_bit,
        " _min_exp =", q._min_exp,
        " _max_exp =", q._max_exp
    )

max_value = None -> need_exponent_sign_bit = 1  _min_exp = -2  _max_exp = 1
max_value = 0.5 -> need_exponent_sign_bit = 0  _min_exp = -4  _max_exp = 3
max_value = 1.0 -> need_exponent_sign_bit = 0  _min_exp = -4  _max_exp = 3
max_value = 2.0 -> need_exponent_sign_bit = 1  _min_exp = -2  _max_exp = 1
max_value = 8.0 -> need_exponent_sign_bit = 1  _min_exp = -2  _max_exp = 1


##Funcion:   quantized_bits
xq = round(x / scale) × scale
scale = 2^(integer − bits + keep_negative) 

In [13]:
import numpy as np
import tensorflow as tf
from qkeras import quantizers

# -------------------------
# CONFIG
# -------------------------
bits = 6
integer = 2
symmetric = 0           # 0 o 1
keep_negative = True   # True (signed) o False (unsigned)
alpha = 1.0             # dejamos fijo para este test
# use_stochastic_rounding = False  # por defecto

# Valores de prueba
x = np.array([-6.0, -2.5, -1.2, -0.8, -0.3, 0.3, 0.8, 1.2, 2.5, 4.0], dtype=np.float32)

# -------------------------
# helpers: redondeo como tf.round (half away from zero)
# -------------------------
def round_half_away_from_zero(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    pos = np.floor(a + 0.5)
    neg = np.ceil(a - 0.5)
    return np.where(a >= 0, pos, neg).astype(np.int32)

# -------------------------
# 1) Crear cuantizador QKeras
# -------------------------
q = quantizers.quantized_bits(
    bits=bits,
    integer=integer,
    symmetric=symmetric,
    keep_negative=keep_negative,
    alpha=alpha
)

print("===================================")
print("QKeras quantized_bits")
print("===================================")
print(f"bits={bits}, integer={integer}, symmetric={symmetric}, keep_negative={keep_negative}, alpha={alpha}")

# -------------------------
# 2) Reglas internas (según QKeras)
#    data_type_scale = 2 ** (integer - bits + keep_negative)
#    (si keep_negative=True, se reserva 1 bit para signo)
# -------------------------
kn = 1 if keep_negative else 0
data_type_scale = 2.0 ** (integer - bits + kn)

# rango de enteros representables (qmin/qmax)
if keep_negative:
    if symmetric:
        # mismo # de niveles + y -
        qmin = -(2 ** (bits - 1) - 1)
        qmax = +(2 ** (bits - 1) - 1)
    else:
        # two's complement típico: un negativo extra
        qmin = -(2 ** (bits - 1))
        qmax = +(2 ** (bits - 1) - 1)
else:
    qmin = 0
    qmax = (2 ** bits) - 1

print("-----------------------------------")
print("Reglas internas:")
print(f"sign_bit (keep_negative) = {kn}")
print(f"data_type_scale = 2^(integer - bits + sign_bit) = 2^({integer} - {bits} + {kn}) = {data_type_scale}")
print(f"qmin = {qmin}")
print(f"qmax = {qmax}")
print("-----------------------------------")

# -------------------------
# 3) Valores posibles (teóricos) que puede representar
# -------------------------
int_vals = np.arange(qmin, qmax + 1, dtype=np.int32)
rep_vals = int_vals.astype(np.float32) * data_type_scale

print(f"Enteros posibles ({len(int_vals)} niveles): {int_vals.tolist()}")
print(f"Valores representables (multiplicando por scale): {rep_vals.tolist()}")
print("-----------------------------------")

# -------------------------
# 4) Cuantización "por fórmula" (orden correcto tipo QKeras, sin stochastic rounding):
#    x_int = clip( round(x / scale), qmin, qmax )
#    x_q   = x_int * scale
# -------------------------
x_scaled = x / data_type_scale
x_int_raw = round_half_away_from_zero(x_scaled)          # round primero
x_int = np.clip(x_int_raw, qmin, qmax).astype(np.int32)  # clip después
xq_formula = x_int.astype(np.float32) * data_type_scale

# Salida real QKeras
xq_qkeras = q(tf.constant(x)).numpy()

print("Entradas x:")
print(x)

print("\nPaso a paso:")
print("x_scaled = x / data_type_scale:")
print(x_scaled)

print("\nx_int_raw = round_half_away_from_zero(x_scaled):")
print(x_int_raw)

print(f"\nx_int = clip(x_int_raw, {qmin}, {qmax}):")
print(x_int)

print("\nx_q (formula) = x_int * data_type_scale:")
print(xq_formula)

print("\nx_q (QKeras):")
print(xq_qkeras)

print("\n¿Coinciden formula y QKeras?")
print(np.allclose(xq_formula, xq_qkeras))
print("===================================")

QKeras quantized_bits
bits=6, integer=2, symmetric=0, keep_negative=True, alpha=1.0
-----------------------------------
Reglas internas:
sign_bit (keep_negative) = 1
data_type_scale = 2^(integer - bits + sign_bit) = 2^(2 - 6 + 1) = 0.125
qmin = -32
qmax = 31
-----------------------------------
Enteros posibles (64 niveles): [-32, -31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Valores representables (multiplicando por scale): [-4.0, -3.875, -3.75, -3.625, -3.5, -3.375, -3.25, -3.125, -3.0, -2.875, -2.75, -2.625, -2.5, -2.375, -2.25, -2.125, -2.0, -1.875, -1.75, -1.625, -1.5, -1.375, -1.25, -1.125, -1.0, -0.875, -0.75, -0.625, -0.5, -0.375, -0.25, -0.125, 0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0, 1.125, 1.25, 1.375, 1.5, 1.625, 1.75, 1.875, 2.0, 2.125, 2.25, 2.3

REF: https://raw.githubusercontent.com/google/qkeras/master/qkeras/quantizers.py